## 3. FEVD

In [ ]:
import pandas as pd
from pathlib import Path

### 3.1 Consolidated Results

In [ ]:
def load_all_csvs(directory_path):
    data_dir = Path(directory_path)
    csv_files = list(data_dir.glob("*.csv"))

    if not csv_files:
        return None

    all_dataframes = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            all_dataframes.append(df)
        except Exception as e:
            print(f"Error loading {file.name}: {e}")

    if all_dataframes:
        return pd.concat(all_dataframes, ignore_index=True)
    return None

file_dir = "../../results/03_forecast_error_variance_decomposition"
FEVD_master = load_all_csvs(file_dir)

In [3]:
# Bringing the "event_date" into a standardized format
FEVD_master["event_date"] = pd.to_datetime(FEVD_master["event_date"], format = "mixed")
FEVD_master["event_date"] = FEVD_master["event_date"].dt.strftime("%Y-%m-%d")
FEVD_master['relationship'] = FEVD_master['source_market'] + ' -> ' + FEVD_master['target_market']

In [4]:
FEVD_master

,event_date,contract,source_market,target_market,lag_minute,FEVD,relationship
0,2024-01-31,0bp,CME,PM,1,0.000009,CME -> PM
1,2024-01-31,0bp,CME,PM,5,0.000839,CME -> PM
2,2024-01-31,0bp,CME,PM,30,0.004766,CME -> PM
3,2024-01-31,0bp,CME,PM,60,0.013682,CME -> PM
4,2024-01-31,0bp,PM,CME,1,0.000002,PM -> CME
...,...,...,...,...,...,...,...
811,2026-04-29,25bp_dec,PM,KAL,60,0.001712,PM -> KAL
812,2026-04-29,25bp_dec,KAL,PM,1,0.000501,KAL -> PM
813,2026-04-29,25bp_dec,KAL,PM,5,0.000941,KAL -> PM
814,2026-04-29,25bp_dec,KAL,PM,30,0.001588,KAL -> PM


In [6]:
fevd_ss = FEVD_master.groupby("relationship")["FEVD"].agg(["count","mean","std"])
fevd_ss = fevd_ss.rename(columns={"count":"total_obs","mean":"avg_fevd","std": "std_fevd"})
fevd_ss.sort_values(["avg_fevd","std_fevd"], ascending = [False,False])


,total_obs,avg_fevd,std_fevd
relationship,,,
KAL -> PM,136,0.004295,0.006270
CME -> KAL,136,0.004150,0.005203
PM -> KAL,136,0.003534,0.004601
CME -> PM,136,0.003086,0.004575
KAL -> CME,136,0.002067,0.004534
PM -> CME,136,0.001277,0.001611


### 3.2 Grouped by lag

In [8]:
fevd_ss = FEVD_master.groupby(['relationship',"lag_minute"])["FEVD"].agg(["count","mean","std"])
fevd_ss = fevd_ss.rename(columns={"count":"total_obs","mean":"avg_fevd","std": "std_fevd"})
fevd_ss= fevd_ss.sort_values(["lag_minute","avg_fevd"], ascending=[True,False])
print(fevd_ss)

                         total_obs  avg_fevd  std_fevd
relationship lag_minute                               
KAL -> PM    1                  34  0.001863  0.002642
PM -> KAL    1                  34  0.001543  0.002147
CME -> KAL   1                  34  0.000860  0.001726
CME -> PM    1                  34  0.000715  0.001225
KAL -> CME   1                  34  0.000349  0.000895
PM -> CME    1                  34  0.000274  0.000425
KAL -> PM    5                  34  0.002896  0.004004
CME -> KAL   5                  34  0.002707  0.003319
PM -> KAL    5                  34  0.002667  0.003230
CME -> PM    5                  34  0.002105  0.003444
KAL -> CME   5                  34  0.000661  0.001111
PM -> CME    5                  34  0.000508  0.000590
CME -> KAL   30                 34  0.005646  0.005632
KAL -> PM    30                 34  0.005511  0.006817
PM -> KAL    30                 34  0.004547  0.005550
CME -> PM    30                 34  0.004097  0.005130
KAL -> CME

### 3.3 Differentiated at Volume

In [ ]:
volume_df = pd.read_csv("../../data/processed/Volume/volume.csv")
volume_df["tritile_CME"] =  pd.qcut(volume_df["CME"], q = 3, labels = ["low","medium","high"])
volume_df["tritile_PM"] = pd.qcut(volume_df["PM"], q =3, labels = ["low","medium","high"] )
volume_df["tritile_Kalshi"] = pd.qcut(volume_df["Kalshi"],q=3, labels =["low", "medium","high"])

volume_info = volume_df[["event_date","contract", "tritile_CME","tritile_PM","tritile_Kalshi"]]
FEVD_merged = pd.merge(FEVD_master,volume_info, on = ["event_date","contract"], how = "left")

#### 3.3.1 CME

In [12]:
CME_vol_df = FEVD_merged[FEVD_merged["source_market"] == "CME"]

In [14]:
fevd_ss = CME_vol_df.groupby(["relationship","tritile_CME"])["FEVD"].agg(["count","mean","std"])
fevd_ss = fevd_ss.rename(columns={"count":"total_obs","mean":"avg_fevd","std": "std_fevd"})
fevd_ss= fevd_ss.sort_values(["avg_fevd","std_fevd"], ascending = [False,False])
print(fevd_ss)

                          total_obs  avg_fevd  std_fevd
relationship tritile_CME                               
CME -> KAL   high                48  0.005707  0.006329
CME -> PM    high                48  0.005508  0.006179
CME -> KAL   low                 48  0.004059  0.004805
             medium              40  0.002392  0.003384
CME -> PM    medium              40  0.002139  0.003119
             low                 48  0.001452  0.002094


/tmp/ipykernel_635/2719196918.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fevd_ss = CME_vol_df.groupby(["relationship","tritile_CME"])["FEVD"].agg(["count","mean","std"])


#### 3.3.2 PM

In [15]:
PM_vol_df = FEVD_merged[FEVD_merged["source_market"] == "PM"]

In [16]:
fevd_ss = PM_vol_df.groupby(["relationship","tritile_PM"])["FEVD"].agg(["count","mean","std"])
fevd_ss = fevd_ss.rename(columns={"count":"total_obs","mean":"avg_fevd","std": "std_fevd"})
fevd_ss = fevd_ss.sort_values(["avg_fevd","std_fevd"], ascending = [False,False])
print(fevd_ss)

                         total_obs  avg_fevd  std_fevd
relationship tritile_PM                               
PM -> KAL    high               48  0.003957  0.004092
             medium             44  0.003801  0.003288
             low                44  0.002807  0.006044
PM -> CME    medium             44  0.001934  0.002196
             low                44  0.001036  0.001225
             high               48  0.000895  0.001030


/tmp/ipykernel_635/3905525318.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fevd_ss = PM_vol_df.groupby(["relationship","tritile_PM"])["FEVD"].agg(["count","mean","std"])


#### 3.3.3 Kalshi

In [17]:
Kalshi_vol_df = FEVD_merged[FEVD_merged["source_market"] == "KAL"]

In [18]:
fevd_ss = Kalshi_vol_df.groupby(["relationship","tritile_Kalshi"])["FEVD"].agg(["count","mean","std"])
fevd_ss = fevd_ss.rename(columns={"count":"total_obs","mean":"avg_fevd","std": "std_fevd"})
fevd_ss = fevd_ss.sort_values(["avg_fevd","std_fevd"], ascending = [False,False])
print(fevd_ss)

                             total_obs  avg_fevd  std_fevd
relationship tritile_Kalshi                               
KAL -> PM    high                   48  0.005615  0.005132
             low                    48  0.005164  0.008703
KAL -> CME   low                    48  0.004167  0.007044
KAL -> PM    medium                 40  0.001668  0.001715
KAL -> CME   high                   48  0.001073  0.001276
             medium                 40  0.000741  0.000987


/tmp/ipykernel_635/1300548441.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  fevd_ss = Kalshi_vol_df.groupby(["relationship","tritile_Kalshi"])["FEVD"].agg(["count","mean","std"])
